# Chapter 21: Hypothesis Tests without Mechanical Thinking
Synthetic NRG experiment data illustrate p-values, effect sizes, permutation tests, and multiplicity.

In [1]:
import math
import random
import statistics
import matplotlib.pyplot as plt
from datasciencebook.hypothesis_tests import bonferroni_alpha, one_sample_z_test, permutation_mean_test, standardized_mean_difference
rng = random.Random(20260829)
control = [rng.gauss(100, 12) for _ in range(40)]
treatment = [rng.gauss(105, 12) for _ in range(40)]
print(f"Control mean: {statistics.mean(control):.2f}")
print(f"Treatment mean: {statistics.mean(treatment):.2f}")

Control mean: 100.63
Treatment mean: 104.91


In [2]:
difference = statistics.mean(treatment) - statistics.mean(control)
standard_error = math.sqrt(statistics.variance(control) / 40 + statistics.variance(treatment) / 40)
z_statistic, p_value = one_sample_z_test(difference, 0, standard_error)
print(f"Mean difference: {difference:.2f}")
print(f"Standard error: {standard_error:.2f}")
print(f"z statistic: {z_statistic:.3f}")
print(f"Two-sided p-value: {p_value:.4f}")

Mean difference: 4.28
Standard error: 2.50
z statistic: 1.713
Two-sided p-value: 0.0867


In [3]:
pooled_sd = math.sqrt((39 * statistics.variance(control) + 39 * statistics.variance(treatment)) / 78)
effect_size = standardized_mean_difference(statistics.mean(treatment), statistics.mean(control), pooled_sd)
print(f"Pooled standard deviation: {pooled_sd:.2f}")
print(f"Standardized mean difference: {effect_size:.3f}")

Pooled standard deviation: 11.18
Standardized mean difference: 0.383


In [4]:
permutation_difference, permutation_p = permutation_mean_test(treatment, control, repetitions=2000, seed=21)
print(f"Permutation difference: {permutation_difference:.2f}")
print(f"Permutation p-value: {permutation_p:.4f}")

Permutation difference: 4.28
Permutation p-value: 0.0865


In [5]:
for tests in [1, 5, 20]:
    print(f"{tests} planned tests: per-test alpha={bonferroni_alpha(0.05, tests):.4f}")

1 planned tests: per-test alpha=0.0500
5 planned tests: per-test alpha=0.0100
20 planned tests: per-test alpha=0.0025


In [6]:
null_rng = random.Random(2100)
null_p_values = []
for _ in range(500):
    first = [null_rng.gauss(100, 12) for _ in range(40)]
    second = [null_rng.gauss(100, 12) for _ in range(40)]
    null_difference = statistics.mean(first) - statistics.mean(second)
    null_se = math.sqrt(statistics.variance(first) / 40 + statistics.variance(second) / 40)
    null_p_values.append(one_sample_z_test(null_difference, 0, null_se)[1])
print(f"False positives at alpha 0.05: {sum(p < 0.05 for p in null_p_values)} of 500")

False positives at alpha 0.05: 29 of 500


In [7]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(null_p_values, bins=10, color="#345995", edgecolor="white")
axes[0].axvline(0.05, color="#C34A36", linestyle="--")
axes[0].set(title="P-values when the null is true", xlabel="p-value", ylabel="Count")
tests = list(range(1, 21))
axes[1].plot(tests, [bonferroni_alpha(0.05, count) for count in tests], color="#25705A")
axes[1].set(title="Bonferroni threshold", xlabel="Number of planned tests", ylabel="Per-test alpha")
fig.tight_layout()
plt.show()

<Figure size 1000x400 with 2 Axes>

## Your turn
Specify a business question, minimum worthwhile effect, hypotheses, error rates, and reporting plan before calculating a p-value.